# 01 · Data audit (Team 10, Deliverable 3)

**Question for this notebook:** what is in the two AA files, at what grain, and what is missing?

**Data mode.** The first output line says `USING FAKE DATA` or `USING REAL DATA`. Until a team
member puts the NDA files in `data/raw/`, everything runs on a generated fake panel with the same
columns (`scripts/make_fake_data.py`). Fake numbers describe the generator, not American Airlines.

**NDA habit, used even on fake data:** this notebook prints counts, shares and metadata only.
It never displays raw rows (`.head()`, samples), so a real run is safe to look at but its
outputs still must not be committed.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

MODELING = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "team-10-modeling"]
                if (p / "src" / "forecasting.py").exists())
sys.path.insert(0, str(MODELING / "src"))

from data import BANNER, MODE, load_airports, load_pax, write_table

print(BANNER)
pax_raw = load_pax()
apt_raw = load_airports()
print(f"passenger file: {len(pax_raw):,} rows x {pax_raw.shape[1]} columns")
print(f"airport file:   {len(apt_raw):,} rows x {apt_raw.shape[1]} columns")

## 1. Columns and what they mean

Meanings follow the data description that ships with the files. Where a meaning is uncertain,
it goes to the PM question list at the end instead of being guessed.

In [ ]:
MEANING = {
    "Unnamed: 0": "Row number written by the export",
    "DEP_MONTH": "Departure month (1-12)",
    "DEP_YEAR": "Departure year",
    "ORG_AIRPRT": "Origin airport code",
    "DES_AIRPRT": "Destination airport code",
    "OD_PAX": "Passengers travelling origin -> destination in that month (TARGET)",
    "OD_REV_USD": "Revenue from those passengers in that month, USD",
    "SF_LOAD_TMS": "Warehouse (Snowflake) load timestamp of the extract",
}
cols = pd.DataFrame({
    "dtype": pax_raw.dtypes.astype(str),
    "n_unique": pax_raw.nunique(),
    "n_missing": pax_raw.isna().sum(),
}).drop(index="DATE", errors="ignore")
cols["meaning"] = cols.index.map(MEANING).fillna("UNDOCUMENTED - ask a PM")
cols

In [ ]:
apt_cols = pd.DataFrame({
    "dtype": apt_raw.dtypes.astype(str),
    "n_unique": apt_raw.nunique(),
    "pct_missing": (100 * apt_raw.isna().mean()).round(1),
})
apt_cols

## 2. Grain and duplicates

Expected grain: one row per origin, destination, year and month.

In [ ]:
grain = ["ORG_AIRPRT", "DES_AIRPRT", "DEP_YEAR", "DEP_MONTH"]
checks = {
    "rows breaking the grain (duplicate route-month)": int(pax_raw.duplicated(grain).sum()),
    "exact duplicate rows": int(pax_raw.duplicated().sum()),
    "Unnamed: 0 equals 0..n-1 in file order": bool((pax_raw["Unnamed: 0"].to_numpy() == np.arange(len(pax_raw))).all()),
    "distinct SF_LOAD_TMS values": int(pax_raw["SF_LOAD_TMS"].nunique()),
    "OD_PAX <= 0": int((pax_raw["OD_PAX"] <= 0).sum()),
    "OD_PAX non-integer (share)": round(float((pax_raw["OD_PAX"] % 1 != 0).mean()), 3),
    "origin == destination": int((pax_raw["ORG_AIRPRT"] == pax_raw["DES_AIRPRT"]).sum()),
    "unparseable dates": int(pax_raw["DATE"].isna().sum()),
    "first month": str(pax_raw["DATE"].min().date()),
    "last month": str(pax_raw["DATE"].max().date()),
    "distinct routes": int(pax_raw.groupby(["ORG_AIRPRT", "DES_AIRPRT"]).ngroups),
    "distinct airports (either end)": int(pd.concat([pax_raw["ORG_AIRPRT"], pax_raw["DES_AIRPRT"]]).nunique()),
}
audit = pd.Series(checks, name="value").to_frame()
audit

A fractional `OD_PAX` share above zero means the counts are estimates (or averaged), not tickets.
That matters for the loss: a count model (Poisson) is still reasonable, but the values are not
strictly integers. It goes to the PM question list.

## 3. Missingness

Missing *cells* and missing *rows* are different problems. A route that did not fly in a month
has no row at all. So we also measure how many route-months are absent between each route's
first and last appearance.

In [ ]:
pax = pax_raw.dropna(subset=["DATE"])
months = pd.date_range(pax["DATE"].min(), pax["DATE"].max(), freq="MS")
span = pax.groupby(["ORG_AIRPRT", "DES_AIRPRT"])["DATE"].agg(["min", "max", "size"])
span["months_spanned"] = ((span["max"].dt.year - span["min"].dt.year) * 12
                          + span["max"].dt.month - span["min"].dt.month + 1)
span["gaps"] = span["months_spanned"] - span["size"]
print(f"route-months present: {span['size'].sum():,}")
print(f"route-months missing inside a route's active span: {span['gaps'].sum():,} "
      f"({span['gaps'].sum() / span['months_spanned'].sum():.1%})")
print(f"routes active in every month: {(span['size'] == len(months)).mean():.1%}")

rows_per_month = pax.groupby("DATE").size()
gap_months = rows_per_month[rows_per_month < 0.8 * rows_per_month.median()]
print("\nmonths with < 80% of the median row count:",
      ", ".join(d.strftime("%Y-%m") for d in gap_months.index) or "none")

In [ ]:
codes = set(pax["ORG_AIRPRT"]) | set(pax["DES_AIRPRT"])
lookup_codes = set(apt_raw["AIRPRT_CD"].dropna())
dup_codes = int(apt_raw["AIRPRT_CD"].duplicated().sum())
state_blank = apt_raw["STATE_PROVNC_CD"].isna()
non_us = apt_raw["CNTRY_CD"].ne("US")
join = {
    "airport codes in passenger file": len(codes),
    "of those, missing from the airport lookup": len(codes - lookup_codes),
    "duplicate codes in the airport lookup": dup_codes,
    "state blank": int(state_blank.sum()),
    "state blank AND country is US": int((state_blank & ~non_us).sum()),
    "country blank": int(apt_raw["CNTRY_CD"].isna().sum()),
}
pd.Series(join, name="value").to_frame()

**Therefore:**

- A blank state on a non-US airport is structural (no US state applies). It is filled with
  `NONE`, not imputed. A blank state on a *US* airport would be a real gap; the count above
  should be 0.
- Missing route-months are **not** filled with zero in training. A missing row may mean "no
  service" or "not recorded", which is an open PM question. In scoring, a route-month we forecast
  that has no actual counts as actual 0, and an actual we did not forecast counts as forecast 0
  (see `src/forecasting.py`).
- Months with sharply fewer rows (expected around spring 2020) motivate the disruption window
  in `02_eda_demand.ipynb`.

In [ ]:
md = ["## Passenger file columns", "", "| column | dtype | n_unique | n_missing | meaning |", "|---|---|---|---|---|"]
md += [f"| {c} | {r.dtype} | {r.n_unique} | {r.n_missing} | {r.meaning} |" for c, r in cols.iterrows()]
md += ["", "## Airport file columns", "", "| column | dtype | n_unique | pct_missing |", "|---|---|---|---|"]
md += [f"| {c} | {r.dtype} | {r.n_unique} | {r.pct_missing} |" for c, r in apt_cols.iterrows()]
md += ["", "## Grain and quality checks", "", "| check | value |", "|---|---|"]
md += [f"| {k} | {v} |" for k, v in checks.items()]
md += ["", "## Airport join", "", "| check | value |", "|---|---|"]
md += [f"| {k} | {v} |" for k, v in join.items()]
md += ["", f"Route-months missing inside active spans: {span['gaps'].sum():,} "
       f"({span['gaps'].sum() / span['months_spanned'].sum():.1%})."]
print("wrote", write_table("data_audit.md", "# Data audit — Team 10 (D3)\n\n_Generated by `notebooks/01_data_audit.ipynb`._\n\n" + "\n".join(md) + "\n"))